# Hunyuan3D 2.1 weights → Kaggle dataset (run once)

Downloads every file the main notebook needs (**17.8 GB**) into `/kaggle/working/hy3d-weights`. You then turn this
notebook's output into a private dataset. After that, attaching the dataset to `hunyuan3d_2_1_kaggle_dual_t4.ipynb`
means nothing gets downloaded again.

**This needs no GPU,** so building it doesn't use any of your weekly GPU hours.

1. **Settings → Accelerator → None**, **Internet → On**
2. **Save Version → Save & Run All (Commit)**, then wait for the version to finish (about 10–20 minutes).
3. Create the dataset from the output. Either:
   - open the finished version, go to the **Output** tab, and choose **New Dataset**, or
   - go to **Datasets → New Dataset**, pick **Notebook Output Files**, and select this notebook.

   Name it, for example, `hunyuan3d-2-1-weights`, and keep it **Private**.
   Already have an incomplete dataset? Open it, choose **New Version**, and pick this notebook's output instead.
4. In the main notebook: **Add Input → Your Datasets → hunyuan3d-2-1-weights.** It's found automatically.

Kaggle's output limit is 20 GB, so don't write anything else to `/kaggle/working` in this notebook.

In [ ]:
!pip install -q --root-user-action=ignore hf_transfer

In [ ]:
%%writefile /tmp/prefetch.py
"""Downloads every weight file (~17.8 GB) into one folder, in parallel.

Usage: python prefetch.py <weights_dir>

Layout (the workers and the Kaggle weights dataset both use it):
  <weights_dir>/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/       shape model      6.9 GB
  <weights_dir>/tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1/  texture model    6.4 GB
  <weights_dir>/facebook/dinov2-giant/                          texture encoder  4.2 GB
  <weights_dir>/RealESRGAN_x4plus.pth                           upscaler
  <weights_dir>/u2net/u2net.onnx                                background removal
"""
import os, shutil, sys, urllib.request
from concurrent.futures import ThreadPoolExecutor

try:
    import hf_transfer  # noqa: F401  (several times faster on Kaggle)
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except ImportError:
    pass
from huggingface_hub import snapshot_download

W = sys.argv[1]
DIT = "tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/model.fp16.ckpt"
PAINT = "tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1/unet/diffusion_pytorch_model.bin"
DINO = "facebook/dinov2-giant/model.safetensors"
ESRGAN = "RealESRGAN_x4plus.pth"
U2NET = "u2net/u2net.onnx"


def have(rel):
    return os.path.exists(os.path.join(W, rel))


def hub(repo_id, patterns):
    # One call per repo: parallel calls sharing a local_dir clobber each other's temp files.
    if patterns:
        snapshot_download(repo_id=repo_id, allow_patterns=patterns, local_dir=os.path.join(W, repo_id))
    return repo_id


def url(link, rel):
    dest = os.path.join(W, rel)
    if not os.path.exists(dest):
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        urllib.request.urlretrieve(link, dest + ".part")
        os.replace(dest + ".part", dest)
    return rel


# Anything already present (e.g. linked in from a partial dataset) is skipped.
tasks = [
    lambda: hub("tencent/Hunyuan3D-2.1", ([] if have(DIT) else ["hunyuan3d-dit-v2-1/*"])
                + ([] if have(PAINT) else ["hunyuan3d-paintpbr-v2-1/*"])),
    lambda: hub("facebook/dinov2-giant", [] if have(DINO) else ["*.json", "*.safetensors", "*.txt"]),
    lambda: url("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth", ESRGAN),
    lambda: url("https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx", U2NET),
]
with ThreadPoolExecutor(len(tasks)) as pool:
    for done in pool.map(lambda task: task(), tasks):
        print("ready:", done)

for repo_id in ("tencent/Hunyuan3D-2.1", "facebook/dinov2-giant"):  # download bookkeeping only
    shutil.rmtree(os.path.join(W, repo_id, ".cache"), ignore_errors=True)

missing = [rel for rel in (DIT, PAINT, DINO, ESRGAN, U2NET) if not have(rel)]
if missing:
    sys.exit(f"Download incomplete, missing: {missing}")
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(W) for f in fs)
print(f"All weights present: {size / 2**30:.1f} GB in {W}")

In [ ]:
import subprocess, sys
# check=True fails the commit if anything is missing, so an incomplete output never becomes a dataset.
subprocess.run([sys.executable, "/tmp/prefetch.py", "/kaggle/working/hy3d-weights"], check=True)
!du -sh /kaggle/working/hy3d-weights/* /kaggle/working/hy3d-weights/*/*